# Arroyo Capital: inversión y portafolio en la industria del cine

**Proyecto final de Programación Analítica II**  
**Estudiante:** Vicente Alvarez  
**Universidad de Montevideo**

El objetivo es analizar la relación entre riesgo y retorno de distintos tipos de películas y construir una tesis de inversión para distribuir USD 500 millones.

## 1. Análisis exploratorio y correcciones de calidad

### 1.1 Carga inicial de los datos

Primero importamos las herramientas necesarias y definimos la ubicación de los archivos. Usamos rutas relativas para que el notebook funcione en otra computadora al clonar el repositorio.

In [1]:
from pathlib import Path
import pandas as pd

# Esta ruta funciona tanto si Jupyter inicia en la raíz como en notebooks/.
RAIZ_PROYECTO = Path.cwd()
if RAIZ_PROYECTO.name == "notebooks":
    RAIZ_PROYECTO = RAIZ_PROYECTO.parent

RUTA_PELICULAS = RAIZ_PROYECTO / "datos" / "peliculas.csv"
RUTA_IPC = RAIZ_PROYECTO / "datos" / "ipc_eeuu.csv"

print(f"Películas: {RUTA_PELICULAS}")
print(f"IPC: {RUTA_IPC}")

Películas: C:\Users\Santi\Documents\Codex\programacion-analitica-2-portafolio-cine-alvarez\datos\peliculas.csv
IPC: C:\Users\Santi\Documents\Codex\programacion-analitica-2-portafolio-cine-alvarez\datos\ipc_eeuu.csv


Declaramos los tipos al cargar los archivos. Las variables de texto usan el tipo `string`, los enteros usan `Int64` y las variables continuas usan `Float64`. Estos tipos admiten valores faltantes, algo importante porque la consigna advierte que la base no está auditada. La fecha se convierte por separado con `errors="coerce"`, de modo que un valor inválido se transforma en faltante en vez de detener todo el notebook.

In [2]:
tipos_peliculas = {
    "id_pelicula": "Int64",
    "titulo": "string",
    "anio_estreno": "Int64",
    "fecha_estreno": "string",
    "genero": "string",
    "clasificacion": "string",
    "pais_origen": "string",
    "es_secuela": "string",
    "director": "string",
    "guionista": "string",
    "protagonista": "string",
    "productora": "string",
    "duracion_minutos": "Float64",
    "puntaje_imdb": "Float64",
    "cantidad_votos": "Float64",
    "presupuesto_usd": "Float64",
    "recaudacion_mundial_usd": "Float64",
}

peliculas = pd.read_csv(RUTA_PELICULAS, dtype=tipos_peliculas)
peliculas["fecha_estreno"] = pd.to_datetime(
    peliculas["fecha_estreno"], errors="coerce"
)

ipc = pd.read_csv(
    RUTA_IPC,
    dtype={"anio": "Int64", "ipc_eeuu": "Float64"},
)

Verificamos que ambos archivos se hayan cargado mostrando sus dimensiones, las primeras filas y los tipos resultantes. En esta instancia solo observamos: todavía no corregimos ni eliminamos datos.

In [3]:
print(f"peliculas.csv: {peliculas.shape[0]:,} filas y {peliculas.shape[1]} columnas")
print(f"ipc_eeuu.csv: {ipc.shape[0]:,} filas y {ipc.shape[1]} columnas")

display(peliculas.head())
display(ipc.head())

peliculas.csv: 7,740 filas y 17 columnas
ipc_eeuu.csv: 41 filas y 2 columnas


,id_pelicula,titulo,anio_estreno,fecha_estreno,genero,clasificacion,pais_origen,es_secuela,director,guionista,protagonista,productora,duracion_minutos,puntaje_imdb,cantidad_votos,presupuesto_usd,recaudacion_mundial_usd
0,15834,Hatchet II,2010,2010-07-01,Acción,Mayores de 17,Estados Unidos,Si,Adam Green,Adam Green,Danielle Harris,ArieScope Pictures,85.0,5.5,13000.0,800000.0,156190.0
1,10992,Shanghai Surprise,1986,1986-08-29,Aventura,Mayores de 13,UK,No,Jim Goddard,John Kohn,Sean Penn,HandMade Films,97.0,3.2,5000.0,17000000.0,2315683.0
2,15201,Mr. Magorium's Wonder Emporium,2007,2007-11-16,Comedia,Apta todo publico,Estados Unidos,No,Zach Helm,Zach Helm,Natalie Portman,Mandate Pictures,93.0,6.2,44000.0,65000000.0,69474661.0
3,13042,Celtic Pride,1996,1996-04-19,Comedia,Mayores de 13,Estados Unidos,No,Tom DeCerchio,Judd Apatow,Damon Wayans,Caravan Pictures,91.0,5.3,9600.0,<NA>,9255027.0
4,11961,Kafka,1991,1991-11-15,Drama,Mayores de 13,Francia,No,Steven Soderbergh,Lem Dobbs,Jeremy Irons,Baltimore Pictures,98.0,6.9,9600.0,11000000.0,1059071.0


,anio,ipc_eeuu
0,1980,82.4
1,1981,90.9
2,1982,96.5
3,1983,99.6
4,1984,103.9


In [4]:
print("Tipos de peliculas.csv:")
display(peliculas.dtypes.to_frame("tipo"))

print("Tipos de ipc_eeuu.csv:")
display(ipc.dtypes.to_frame("tipo"))

Tipos de peliculas.csv:


,tipo
id_pelicula,Int64
titulo,string
anio_estreno,Int64
fecha_estreno,datetime64[us]
genero,string
clasificacion,string
pais_origen,string
es_secuela,string
director,string
guionista,string


Tipos de ipc_eeuu.csv:


,tipo
anio,Int64
ipc_eeuu,Float64


**Conclusión de la carga:** los archivos quedaron disponibles para el análisis. `peliculas.csv` contiene 7.740 registros y 17 variables, mientras que `ipc_eeuu.csv` contiene 41 años y 2 variables. La carga conserva los faltantes y convierte las fechas inválidas en valores ausentes para auditarlas en los siguientes puntos.